# Verify the Scientific Reports analyses (scripts 17 to 19)

**What this does:** runs every new analysis in the manuscript from scratch, then compares each number with the value used in the manuscript and flags any difference.

**What you do:**
1. `Runtime → Change runtime type` → **CPU** (not GPU, so the numbers are comparable).
2. `Runtime → Run all`.
3. When asked, allow Google Drive access (so a disconnect does not lose work), and if asked, upload `uci_baseline.csv` from `ENRICHMENT/05_data/processed/`.
4. Keep this tab open. It takes about **1.5 hours**. If Colab disconnects, just `Run all` again: finished runs are saved and skipped.
5. At the end, read the verdict in the last cell, and send me the file `verify_comparison.csv` it saves in your Drive folder `SciRep_verify`.

Small differences in the third decimal can come from a different xgboost or scikit-learn version. The last cell tells you which ones matter.

In [ ]:
# 1. Packages
!pip install -q imbalanced-learn ucimlrepo xgboost scikit-learn
import xgboost, sklearn, imblearn, numpy, pandas, platform
print("python", platform.python_version(), "| xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__,
      "| imbalanced-learn", imblearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)

In [ ]:
# 2. Working folder (in Google Drive if allowed, so results survive a disconnect)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/SciRep_verify'
except Exception as e:
    print('Drive not mounted, using local folder:', e)
    WORK = '/content/SciRep_verify'
for sub in ('code', 'data', 'data_ext', 'results'):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
print('Working folder:', WORK)

In [ ]:
# 3. Write the analysis scripts (identical to the files in 10_SciRep_submission/code)
CODE = {
"mech.py": "\"\"\"\nmech.py - shared functions for the mechanism analyses (scripts 17-19).\n\nThe model, encoding, scaling and fold construction follow 13_leakage_factorial.py and\n16_factorial_multiseed.py. What is added here is book-keeping: every out-of-fold\nprediction is tagged as belonging to a REAL record or to a record CREATED BY RESAMPLING,\nso that the pooled AUROC of a pre-split-resampling pipeline can be split exactly into\n\n    AUROC_pooled = pi_real * AUROC_real + pi_new * AUROC_new\n\nwhere  AUROC_real = real positives      vs real negatives\n       AUROC_new  = resampled positives vs real negatives\n       pi_new     = share of positives in the evaluated set that were created by resampling\n\nThe identity is exact because AUROC is the probability that a randomly chosen positive is\nranked above a randomly chosen negative, and every positive is either real or created.\n\"\"\"\nimport warnings\nimport numpy as np\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score\nfrom imblearn.over_sampling import SMOTE\nfrom xgboost import XGBClassifier\n\nwarnings.filterwarnings(\"ignore\")\nN_SPLITS = 5\n\n\ndef make_xgb(pos_weight, seed, n_jobs=2):\n    \"\"\"Hyperparameters copied verbatim from 13_leakage_factorial.py (CPU hist).\"\"\"\n    return XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8,\n                         colsample_bytree=0.7, min_child_weight=5, reg_lambda=2.0,\n                         scale_pos_weight=pos_weight, eval_metric=\"logloss\",\n                         random_state=seed, tree_method=\"hist\", n_jobs=n_jobs)\n\n\ndef make_lr(balanced, seed):\n    return LogisticRegression(C=1.0, max_iter=3000, solver=\"lbfgs\",\n                              class_weight=\"balanced\" if balanced else None)\n\n\ndef integer_columns(X):\n    \"\"\"Columns in which every real record takes an integer value (binary, one-hot, counts).\"\"\"\n    return np.all(np.mod(X, 1) == 0, axis=0)\n\n\ndef binary_columns(X):\n    \"\"\"Columns in which every real record is 0 or 1 (binary and one-hot indicators).\"\"\"\n    return np.all((X == 0) | (X == 1), axis=0)\n\n\ndef resample(X, y, method, seed, k=5):\n    \"\"\"Return (Xr, yr, is_new). Real rows always come first, in their original order.\"\"\"\n    n = len(y)\n    if method == \"none\":\n        return X, y, np.zeros(n, bool)\n    if method in (\"smote\", \"smote_round\", \"smote_round_bin\"):\n        Xr, yr = SMOTE(random_state=seed, k_neighbors=k).fit_resample(X, y)\n        assert np.array_equal(Xr[:n], X) and np.array_equal(yr[:n], y)\n        if method in (\"smote_round\", \"smote_round_bin\"):\n            ic = integer_columns(X) if method == \"smote_round\" else binary_columns(X)\n            Xr = Xr.copy()\n            Xr[n:, ic] = np.rint(Xr[n:, ic])\n    elif method == \"ros\":\n        rng = np.random.RandomState(seed)\n        pos = np.flatnonzero(y == 1)\n        need = int((y == 0).sum() - (y == 1).sum())\n        pick = rng.choice(pos, size=need, replace=True)\n        Xr = np.vstack([X, X[pick]])\n        yr = np.concatenate([y, np.ones(need, int)])\n    else:\n        raise ValueError(method)\n    is_new = np.zeros(len(yr), bool)\n    is_new[n:] = True\n    return Xr, yr, is_new\n\n\ndef oof_predict(Xr, yr, gr, grouped, learner, balanced_by_weight, seed, n_jobs=2):\n    if grouped:\n        splits = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=seed).split(\n            np.zeros(len(yr)), yr, gr)\n    else:\n        splits = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed).split(\n            np.zeros(len(yr)), yr)\n    oof = np.zeros(len(yr))\n    for tr, te in splits:\n        sc = StandardScaler().fit(Xr[tr])\n        if learner == \"xgb\":\n            pw = (yr[tr] == 0).sum() / max((yr[tr] == 1).sum(), 1) if balanced_by_weight else 1.0\n            # 13_leakage_factorial.py uses the full-sample ratio; the fold ratio is identical\n            # to three decimals under stratification\n            m = make_xgb(pw, seed, n_jobs)\n        else:\n            m = make_lr(balanced_by_weight, seed)\n        m.fit(sc.transform(Xr[tr]), yr[tr])\n        oof[te] = m.predict_proba(sc.transform(Xr[te]))[:, 1]\n    return oof\n\n\ndef decompose(yr, oof, is_new):\n    \"\"\"Pooled AUROC and its exact split into real and resampled positives.\"\"\"\n    neg = yr == 0\n    real_pos = (yr == 1) & ~is_new\n    new_pos = (yr == 1) & is_new\n    out = dict(auroc_pooled=roc_auc_score(yr, oof))\n    m = neg | real_pos\n    out[\"auroc_real\"] = roc_auc_score(yr[m], oof[m])\n    if new_pos.any():\n        m2 = neg | new_pos\n        out[\"auroc_new\"] = roc_auc_score(yr[m2], oof[m2])\n        out[\"pi_new\"] = new_pos.sum() / (new_pos.sum() + real_pos.sum())\n        out[\"identity\"] = (1 - out[\"pi_new\"]) * out[\"auroc_real\"] + out[\"pi_new\"] * out[\"auroc_new\"]\n    else:\n        out[\"auroc_new\"] = np.nan; out[\"pi_new\"] = 0.0; out[\"identity\"] = out[\"auroc_real\"]\n    return out\n\n\ndef run(X, y, g, method, learner, seed, grouped=True, k=5, n_jobs=2):\n    \"\"\"One pipeline. g may be None (no repeated records); new rows get unique group ids,\n    exactly as in 13_leakage_factorial.py.\"\"\"\n    Xr, yr, is_new = resample(X, y, method, seed, k)\n    if g is None:\n        g = np.arange(len(y))\n    g = np.asarray(g)\n    _, g_int = np.unique(g, return_inverse=True)\n    gr = np.concatenate([g_int, np.arange(g_int.max() + 1, g_int.max() + 1 + is_new.sum())])\n    oof = oof_predict(Xr, yr, gr, grouped, learner, method == \"none\", seed, n_jobs)\n    res = decompose(yr, oof, is_new)\n    res.update(method=method, learner=learner, seed=seed, grouped=grouped,\n               n_real=len(y), n_new=int(is_new.sum()), prevalence=float(y.mean()))\n    return res\n\n\ndef off_support(X, y, seed, k=5):\n    \"\"\"How recognisable are SMOTE rows? Share with >=1 value no real record can take.\"\"\"\n    n = len(y)\n    Xr, yr, is_new = resample(X, y, \"smote\", seed, k)\n    ic = integer_columns(X)\n    frac = np.mod(Xr[n:][:, ic], 1) != 0\n    per_row = frac.sum(1)\n    return dict(n_new=int(is_new.sum()), n_integer_cols=int(ic.sum()), n_cols=int(X.shape[1]),\n                share_rows_off_support=float((per_row > 0).mean()),\n                median_off_support_features=float(np.median(per_row)),\n                share_real_rows_off_support=0.0)\n\n\ndef run_postsplit(X, y, g, mode, learner, seed, k=5, n_jobs=2):\n    \"\"\"Resampling AFTER the patient-grouped split, so that nothing crosses the boundary.\n\n    mode = \"train_only\"       SMOTE inside each training fold; test fold left real\n                              (the correct way to use SMOTE)\n    mode = \"both_separately\"  SMOTE inside each training fold AND, independently, inside\n                              each test fold. No test record, real or synthetic, shares a\n                              parent with any training record, so there is no leakage in\n                              the usual sense - but the evaluated set again contains\n                              synthetic positives.\n    \"\"\"\n    if g is None:\n        g = np.arange(len(y))\n    splits = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=seed).split(\n        np.zeros(len(y)), y, g)\n    ys, ps, news = [], [], []\n    for f, (tr, te) in enumerate(splits):\n        Xtr, ytr = SMOTE(random_state=seed, k_neighbors=k).fit_resample(X[tr], y[tr])\n        if mode == \"both_separately\":\n            Xte, yte = SMOTE(random_state=seed + 1000 + f, k_neighbors=k).fit_resample(X[te], y[te])\n        else:\n            Xte, yte = X[te], y[te]\n        is_new = np.zeros(len(yte), bool); is_new[len(te):] = True\n        sc = StandardScaler().fit(Xtr)\n        m = make_xgb(1.0, seed, n_jobs) if learner == \"xgb\" else make_lr(False, seed)\n        m.fit(sc.transform(Xtr), ytr)\n        ys.append(yte); ps.append(m.predict_proba(sc.transform(Xte))[:, 1]); news.append(is_new)\n    yr, oof, is_new = np.concatenate(ys), np.concatenate(ps), np.concatenate(news)\n    res = decompose(yr, oof, is_new)\n    res.update(method=\"postsplit_\" + mode, learner=learner, seed=seed, grouped=True,\n               n_real=len(y), n_new=int(is_new.sum()), prevalence=float(y.mean()))\n    return res\n",
"17_mechanism_primary.py": "\"\"\"\nSTEP 17 - Where does the inflation live? (Diabetes 130-US Hospitals)\n\nExtends leakage_mechanism (rescoring on real encounters) by tagging every out-of-fold\nprediction as real or created-by-resampling and reporting, for each pre-split resampler\nand each learner:\n\n    AUROC_pooled, AUROC_real, AUROC_new, pi_new, and the identity check.\n\nResamplers (all applied to the FULL dataset before the split, patient-grouped folds):\n    smote        SMOTE, k=5 (the error studied in the paper)\n    smote_round  the same synthetic rows with integer-valued features rounded back to\n                 integers, so that no single feature value is impossible for a real record\n    ros          random oversampling (exact copies of real minority records)\nLearners:\n    xgb          the fixed XGBoost configuration of the paper\n    lr           L2 logistic regression (C=1)\n\nUsage:  python 17_mechanism_primary.py <uci_baseline.csv> <out_dir> [seed ...]\n\"\"\"\nimport sys, os, json, time\nimport numpy as np, pandas as pd\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport mech\n\nTARGET, GROUP = \"readmitted\", \"patient_nbr\"\nCAT_AS_STR = [\"admission_type_id\", \"discharge_disposition_id\", \"admission_source_id\", \"race\"]\n\n\ndef load_frame(path):\n    \"\"\"Verbatim from 13_leakage_factorial.py.\"\"\"\n    df = pd.read_csv(path, low_memory=False)\n    if df[TARGET].dtype == object:\n        df[TARGET] = (df[TARGET].astype(str) == \"<30\").astype(int)\n    y = df[TARGET].astype(int).values; g = df[GROUP].values\n    X = df.drop(columns=[TARGET, GROUP])\n    for c in CAT_AS_STR:\n        if c in X.columns: X[c] = X[c].astype(str)\n    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]\n    return pd.get_dummies(X, columns=cat, dummy_na=False).astype(float), y, g\n\n\nif __name__ == \"__main__\":\n    path, out_dir = sys.argv[1], sys.argv[2]\n    seeds = [int(s) for s in sys.argv[3:]] or [42, 1, 2]\n    os.makedirs(out_dir, exist_ok=True)\n    Xdf, y, g = load_frame(path)\n    X = Xdf.values.astype(float)\n    assert X.shape == (98490, 155) and y.sum() == 11271 and len(np.unique(g)) == 69311\n    out_csv = os.path.join(out_dir, \"mechanism_primary.csv\")\n    rows = pd.read_csv(out_csv).to_dict(\"records\") if os.path.exists(out_csv) else []\n    done = {(r[\"method\"], r[\"learner\"], r[\"seed\"]) for r in rows}\n\n    with open(os.path.join(out_dir, \"off_support_primary.json\"), \"w\") as fh:\n        json.dump(mech.off_support(X, y, seeds[0]), fh, indent=1)\n\n    for seed in seeds:\n        for learner in (\"xgb\", \"lr\"):\n            for method in (\"none\", \"smote\", \"smote_round\", \"ros\"):\n                if (method, learner, seed) in done: continue\n                t = time.time()\n                r = mech.run(X, y, g, method, learner, seed, grouped=True)\n                r[\"seconds\"] = round(time.time() - t, 1)\n                rows.append(r)\n                pd.DataFrame(rows).to_csv(out_csv, index=False)\n                print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, flush=True)\n",
"17b_mechanism_postsplit.py": "\"\"\"\nSTEP 17b - Is leakage needed at all? (Diabetes 130-US Hospitals)\n\nRuns SMOTE strictly after the patient-grouped split, either in the training folds only\n(correct use) or independently in the training folds and in the test folds. In the second\ncase no information crosses the validation boundary, yet synthetic positives are again\npart of what is scored.\n\nUsage:  python 17b_mechanism_postsplit.py <uci_baseline.csv> <out_dir> [seed ...]\n\"\"\"\nimport sys, os, time, importlib.util\nimport numpy as np, pandas as pd\nhere = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\nimport mech\nspec = importlib.util.spec_from_file_location(\"s17\", os.path.join(here, \"17_mechanism_primary.py\"))\ns17 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s17)\n\nif __name__ == \"__main__\":\n    path, out_dir = sys.argv[1], sys.argv[2]\n    seeds = [int(s) for s in sys.argv[3:]] or [42, 1, 2]\n    Xdf, y, g = s17.load_frame(path); X = Xdf.values.astype(float)\n    out_csv = os.path.join(out_dir, \"mechanism_postsplit.csv\")\n    rows = pd.read_csv(out_csv).to_dict(\"records\") if os.path.exists(out_csv) else []\n    done = {(r[\"method\"], r[\"learner\"], r[\"seed\"]) for r in rows}\n    for seed in seeds:\n        for learner in (\"xgb\", \"lr\"):\n            for mode in (\"train_only\", \"both_separately\"):\n                if (\"postsplit_\" + mode, learner, seed) in done: continue\n                t = time.time()\n                r = mech.run_postsplit(X, y, g, mode, learner, seed)\n                r[\"seconds\"] = round(time.time() - t, 1)\n                rows.append(r); pd.DataFrame(rows).to_csv(out_csv, index=False)\n                print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, flush=True)\n",
"17c_round_binary.py": "\"\"\"\nSTEP 17c - Which features give synthetic records away? (Diabetes 130-US Hospitals)\nPre-split SMOTE with ONLY the binary / one-hot indicator columns rounded back to 0 or 1;\ncount-valued columns keep their interpolated (fractional) values.\nUsage:  python 17c_round_binary.py <uci_baseline.csv> <out_dir> [seed ...]\n\"\"\"\nimport sys, os, time, importlib.util\nimport numpy as np, pandas as pd\nhere = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\nimport mech\nspec = importlib.util.spec_from_file_location(\"s17\", os.path.join(here, \"17_mechanism_primary.py\"))\ns17 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s17)\nif __name__ == \"__main__\":\n    path, out_dir = sys.argv[1], sys.argv[2]\n    seeds = [int(s) for s in sys.argv[3:]] or [42, 1, 2]\n    Xdf, y, g = s17.load_frame(path); X = Xdf.values.astype(float)\n    print(\"binary columns:\", int(mech.binary_columns(X).sum()), \"of\", X.shape[1])\n    rows = []\n    for seed in seeds:\n        r = mech.run(X, y, g, \"smote_round_bin\", \"xgb\", seed, grouped=True); rows.append(r)\n        pd.DataFrame(rows).to_csv(os.path.join(out_dir, \"mechanism_round_binary.csv\"), index=False)\n        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, flush=True)\n",
"18a_fetch_datasets.py": "\"\"\"\nSTEP 18a - Download the six replication datasets from their public repositories.\n\nWrites one CSV per dataset (features plus outcome columns, exactly as distributed) and a\nsmall metadata file. Nothing is filtered or recoded here; that happens in step 18.\n\nNeeds:  pip install ucimlrepo scikit-learn pandas\nUsage:  python 18a_fetch_datasets.py <data_dir>\n\"\"\"\nimport sys, os, json\nimport pandas as pd\nfrom ucimlrepo import fetch_ucirepo\nfrom sklearn.datasets import fetch_openml\n\nUCI = [(\"cdc_dhi\", 891), (\"actg175\", 890), (\"thoracic\", 277), (\"ctg\", 193), (\"heart_failure\", 519)]\n\nif __name__ == \"__main__\":\n    out = sys.argv[1]; os.makedirs(out, exist_ok=True); meta = {}\n    for name, uci_id in UCI:\n        d = fetch_ucirepo(id=uci_id)\n        df = pd.concat([d.data.features, d.data.targets], axis=1)\n        df.to_csv(os.path.join(out, name + \".csv\"), index=False)\n        meta[name] = dict(source=\"UCI Machine Learning Repository\", id=uci_id, name=d.metadata.get(\"name\"),\n                          doi=d.metadata.get(\"dataset_doi\"), rows=len(df), missing=int(df.isna().sum().sum()))\n        print(name, df.shape)\n    m = fetch_openml(data_id=310, as_frame=True, parser=\"auto\")\n    m.frame.to_csv(os.path.join(out, \"mammography.csv\"), index=False)\n    meta[\"mammography\"] = dict(source=\"OpenML\", id=310, rows=len(m.frame), missing=int(m.frame.isna().sum().sum()))\n    print(\"mammography\", m.frame.shape)\n    json.dump(meta, open(os.path.join(out, \"meta.json\"), \"w\"), indent=1)\n",
"18_replication_datasets.py": "\"\"\"\nSTEP 18 - Does the decomposition hold on other public clinical datasets?\n\nFive additional public datasets with no missing values (so that no imputation step can\nitself leak), spanning sample size, imbalance and feature type. Each is run through the\nsame pipelines as the primary dataset with the same two learners and no tuning.\n\n    cdc_dhi        CDC Diabetes Health Indicators (UCI 891)    outcome: Diabetes_binary\n    actg175        AIDS Clinical Trials Group Study 175 (UCI 890)  outcome: cid\n                   baseline covariates only: time, offtrt, cd420 and cd820 are measured\n                   after baseline and are removed; zprior is constant and is removed\n    thoracic       Thoracic Surgery Data (UCI 277)              outcome: Risk1Yr (death within 1 year)\n    ctg            Cardiotocography (UCI 193)                   outcome: NSP = 3 (pathologic)\n                   the CLASS pattern label is a second target and is not used as a feature\n    heart_failure  Heart Failure Clinical Records (UCI 519)     outcome: death_event\n                   follow-up time is removed (it is not known at baseline)\n    mammography    Mammography (OpenML 310)                     outcome: class = 1\n\nUsage:  python 18_replication_datasets.py <data_dir> <out_dir>\n\"\"\"\nimport sys, os, time, json\nimport numpy as np, pandas as pd\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport mech\n\n\ndef load(name, d):\n    df = pd.read_csv(os.path.join(d, name + \".csv\"))\n    if name == \"cdc_dhi\":\n        y = df.pop(\"Diabetes_binary\").astype(int).values\n    elif name == \"actg175\":\n        y = df.pop(\"cid\").astype(int).values\n        df = df.drop(columns=[\"time\", \"offtrt\", \"cd420\", \"cd820\", \"zprior\"])\n    elif name == \"thoracic\":\n        y = (df.pop(\"Risk1Yr\").astype(str) == \"T\").astype(int).values\n        for c in df.columns:\n            if set(df[c].astype(str).unique()) <= {\"T\", \"F\"}:\n                df[c] = (df[c].astype(str) == \"T\").astype(int)\n        cat = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]\n        df = pd.get_dummies(df, columns=cat, dummy_na=False)\n    elif name == \"ctg\":\n        y = (df.pop(\"NSP\").astype(int) == 3).astype(int).values\n        df = df.drop(columns=[\"CLASS\"])\n    elif name == \"heart_failure\":\n        y = df.pop(\"death_event\").astype(int).values\n        df = df.drop(columns=[\"time\"])\n    elif name == \"mammography\":\n        y = (df.pop(\"class\").astype(str).str.strip(\"'\") == \"1\").astype(int).values\n    X = df.astype(float).values\n    assert not np.isnan(X).any()\n    return X, y\n\n\nSEEDS = {\"cdc_dhi\": [42], \"default\": [42, 1, 2, 3, 4]}\nMETHODS = [\"none\", \"smote\", \"smote_round\", \"ros\", \"postsplit_both\"]\n\nif __name__ == \"__main__\":\n    d, out_dir = sys.argv[1], sys.argv[2]\n    names = sys.argv[3:] or [\"thoracic\", \"heart_failure\", \"actg175\", \"ctg\", \"mammography\", \"cdc_dhi\"]\n    out_csv = os.path.join(out_dir, \"replication_datasets.csv\")\n    rows = pd.read_csv(out_csv).to_dict(\"records\") if os.path.exists(out_csv) else []\n    done = {(r[\"dataset\"], r[\"method\"], r[\"learner\"], r[\"seed\"]) for r in rows}\n    desc = {}\n    for name in names:\n        X, y = load(name, d)\n        ic, bc = mech.integer_columns(X), mech.binary_columns(X)\n        os_ = mech.off_support(X, y, 42)\n        desc[name] = dict(n=len(y), events=int(y.sum()), prevalence=float(y.mean()), p=X.shape[1],\n                          integer_cols=int(ic.sum()), binary_cols=int(bc.sum()),\n                          share_new_rows_off_support=os_[\"share_rows_off_support\"])\n        print(name, desc[name], flush=True)\n        for seed in SEEDS.get(name, SEEDS[\"default\"]):\n            for learner in (\"xgb\", \"lr\"):\n                for method in METHODS:\n                    stored = \"postsplit_both_separately\" if method == \"postsplit_both\" else method\n                    if (name, stored, learner, seed) in done: continue\n                    t = time.time()\n                    if method == \"postsplit_both\":\n                        r = mech.run_postsplit(X, y, None, \"both_separately\", learner, seed)\n                    else:\n                        r = mech.run(X, y, None, method, learner, seed, grouped=True)\n                    r[\"dataset\"] = name; r[\"seconds\"] = round(time.time() - t, 1)\n                    rows.append(r); pd.DataFrame(rows).to_csv(out_csv, index=False)\n        sub = pd.DataFrame([r for r in rows if r[\"dataset\"] == name])\n        print(sub.groupby([\"learner\", \"method\"])[[\"auroc_pooled\", \"auroc_real\", \"auroc_new\", \"pi_new\"]]\n              .mean().round(4).to_string(), flush=True)\n        old = json.load(open(os.path.join(out_dir, \"replication_descriptives.json\"))) \\\n            if os.path.exists(os.path.join(out_dir, \"replication_descriptives.json\")) else {}\n        old.update(desc)\n        json.dump(old, open(os.path.join(out_dir, \"replication_descriptives.json\"), \"w\"), indent=1)\n",
"19_simulation.py": "\"\"\"\nSTEP 19 - Simulation: when does resampling before the split inflate AUROC, and why?\n\nData with a KNOWN discrimination are generated from a logistic model with 20 independent\npredictors and an event rate near 10%. The same latent data are presented to the learner\neither as continuous predictors or as binary predictors (each dichotomised at zero), so\nthat the only thing that changes is whether an interpolated record can take a value that\nno real record takes.\n\nPipelines:  correct (no resampling) | SMOTE before the split | SMOTE separately in the\n            training folds and in the test folds (nothing crosses the boundary)\nLearners:   XGBoost (the fixed configuration of the paper) | logistic regression\nSizes:      n = 1,000 and n = 10,000\nReplicates: 10 per scenario\n\nUsage:  python 19_simulation.py <out_dir> [n_reps]\n\"\"\"\nimport sys, os, time\nimport numpy as np, pandas as pd\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport mech\nfrom sklearn.metrics import roc_auc_score\n\nP = 20\n\n\ndef generate(n, kind, rng):\n    z = rng.standard_normal((n, P))\n    X = z if kind == \"continuous\" else (z > 0).astype(float)\n    signal = z if kind == \"continuous\" else 2.0 * X - 1.0\n    beta = np.full(P, 0.17)\n    eta = -2.35 + signal @ beta\n    p = 1.0 / (1.0 + np.exp(-eta))\n    y = (rng.uniform(size=n) < p).astype(int)\n    return X, y, roc_auc_score(y, eta)\n\n\nif __name__ == \"__main__\":\n    out_dir = sys.argv[1]; reps = int(sys.argv[2]) if len(sys.argv) > 2 else 10\n    out_csv = os.path.join(out_dir, \"simulation.csv\")\n    rows = pd.read_csv(out_csv).to_dict(\"records\") if os.path.exists(out_csv) else []\n    done = {(r[\"n\"], r[\"kind\"], r[\"rep\"], r[\"learner\"], r[\"method\"]) for r in rows}\n    for n in (1000, 10000):\n        for kind in (\"continuous\", \"binary\"):\n            for rep in range(reps):\n                rng = np.random.default_rng(100000 * (kind == \"binary\") + 1000 * rep + n)\n                X, y, true_auc = generate(n, kind, rng)\n                for learner in (\"xgb\", \"lr\"):\n                    for method in (\"none\", \"smote\", \"postsplit_both_separately\"):\n                        if (n, kind, rep, learner, method) in done: continue\n                        if method == \"postsplit_both_separately\":\n                            r = mech.run_postsplit(X, y, None, \"both_separately\", learner, rep)\n                        else:\n                            r = mech.run(X, y, None, method, learner, rep, grouped=True)\n                        r.update(n=n, kind=kind, rep=rep, true_auroc=true_auc)\n                        rows.append(r); pd.DataFrame(rows).to_csv(out_csv, index=False)\n            df = pd.DataFrame(rows); s = df[(df.n == n) & (df.kind == kind)]\n            print(n, kind, \"true AUROC %.3f prevalence %.3f\" % (s.true_auroc.mean(), s.prevalence.mean()))\n            print(s.groupby([\"learner\", \"method\"])[[\"auroc_pooled\", \"auroc_real\", \"auroc_new\"]]\n                  .mean().round(3).to_string(), flush=True)\n"
}
for name, src in CODE.items():
    with open(os.path.join(WORK, 'code', name), 'w') as fh:
        fh.write(src)
print('wrote', sorted(CODE))

In [ ]:
# 4. The processed cohort file
import glob, shutil
DATA = os.path.join(WORK, 'data', 'uci_baseline.csv')
if not os.path.exists(DATA):
    hits = glob.glob('/content/drive/MyDrive/**/uci_baseline.csv', recursive=True) if os.path.exists('/content/drive') else []
    if hits:
        print('found in Drive:', hits[0]); shutil.copy(hits[0], DATA)
    else:
        from google.colab import files
        print('Upload uci_baseline.csv (ENRICHMENT/05_data/processed/uci_baseline.csv)')
        up = files.upload()
        shutil.move(list(up)[0], DATA)
import pandas as pd
_d = pd.read_csv(DATA, usecols=['readmitted', 'patient_nbr'])
print(len(_d), 'encounters,', _d.patient_nbr.nunique(), 'patients')

In [ ]:
# 5. Primary dataset: decomposition and probes (about 50 minutes)
%cd {WORK}
!python code/17_mechanism_primary.py data/uci_baseline.csv results 42 1 2
!python code/17b_mechanism_postsplit.py data/uci_baseline.csv results 42 1 2
!python code/17c_round_binary.py data/uci_baseline.csv results 42 1 2

In [ ]:
# 6. Six replication datasets (about 20 minutes)
%cd {WORK}
!python code/18a_fetch_datasets.py data_ext
!python code/18_replication_datasets.py data_ext results

In [ ]:
# 7. Simulation (about 15 minutes)
%cd {WORK}
!python code/19_simulation.py results 10

In [ ]:
# 8. Compare with the manuscript
EXPECTED = {
"diabetes130|lr|none|auroc_pooled": 0.6604,
"diabetes130|lr|none|auroc_real": 0.6604,
"diabetes130|lr|ros|auroc_pooled": 0.666,
"diabetes130|lr|ros|auroc_real": 0.6653,
"diabetes130|lr|ros|auroc_new": 0.6661,
"diabetes130|lr|smote|auroc_pooled": 0.681,
"diabetes130|lr|smote|auroc_real": 0.6559,
"diabetes130|lr|smote|auroc_new": 0.6847,
"diabetes130|lr|smote_round|auroc_pooled": 0.6714,
"diabetes130|lr|smote_round|auroc_real": 0.6568,
"diabetes130|lr|smote_round|auroc_new": 0.6736,
"diabetes130|xgb|none|auroc_pooled": 0.6701,
"diabetes130|xgb|none|auroc_real": 0.6701,
"diabetes130|xgb|ros|auroc_pooled": 0.6959,
"diabetes130|xgb|ros|auroc_real": 0.6935,
"diabetes130|xgb|ros|auroc_new": 0.6962,
"diabetes130|xgb|smote|auroc_pooled": 0.957,
"diabetes130|xgb|smote|auroc_real": 0.6683,
"diabetes130|xgb|smote|auroc_new": 0.9998,
"diabetes130|xgb|smote_round|auroc_pooled": 0.7099,
"diabetes130|xgb|smote_round|auroc_real": 0.6847,
"diabetes130|xgb|smote_round|auroc_new": 0.7136,
"diabetes130|lr|postsplit_both_separately|auroc_pooled": 0.6736,
"diabetes130|lr|postsplit_both_separately|auroc_real": 0.651,
"diabetes130|lr|postsplit_both_separately|auroc_new": 0.677,
"diabetes130|lr|postsplit_train_only|auroc_pooled": 0.651,
"diabetes130|lr|postsplit_train_only|auroc_real": 0.651,
"diabetes130|xgb|postsplit_both_separately|auroc_pooled": 0.9567,
"diabetes130|xgb|postsplit_both_separately|auroc_real": 0.6654,
"diabetes130|xgb|postsplit_both_separately|auroc_new": 0.9999,
"diabetes130|xgb|postsplit_train_only|auroc_pooled": 0.6654,
"diabetes130|xgb|postsplit_train_only|auroc_real": 0.6654,
"diabetes130|xgb|smote_round_bin|auroc_pooled": 0.9534,
"diabetes130|xgb|smote_round_bin|auroc_real": 0.6604,
"diabetes130|xgb|smote_round_bin|auroc_new": 0.9969,
"actg175|lr|none|auroc_pooled": 0.6919,
"actg175|lr|none|auroc_real": 0.6919,
"actg175|lr|postsplit_both_separately|auroc_pooled": 0.7028,
"actg175|lr|postsplit_both_separately|auroc_real": 0.6903,
"actg175|lr|postsplit_both_separately|auroc_new": 0.7088,
"actg175|lr|ros|auroc_pooled": 0.7072,
"actg175|lr|ros|auroc_real": 0.7018,
"actg175|lr|ros|auroc_new": 0.7098,
"actg175|lr|smote|auroc_pooled": 0.7161,
"actg175|lr|smote|auroc_real": 0.7028,
"actg175|lr|smote|auroc_new": 0.7224,
"actg175|lr|smote_round|auroc_pooled": 0.7101,
"actg175|lr|smote_round|auroc_real": 0.7025,
"actg175|lr|smote_round|auroc_new": 0.7137,
"actg175|xgb|none|auroc_pooled": 0.652,
"actg175|xgb|none|auroc_real": 0.652,
"actg175|xgb|postsplit_both_separately|auroc_pooled": 0.8802,
"actg175|xgb|postsplit_both_separately|auroc_real": 0.6583,
"actg175|xgb|postsplit_both_separately|auroc_new": 0.9856,
"actg175|xgb|ros|auroc_pooled": 0.8368,
"actg175|xgb|ros|auroc_real": 0.8019,
"actg175|xgb|ros|auroc_new": 0.8534,
"actg175|xgb|smote|auroc_pooled": 0.8949,
"actg175|xgb|smote|auroc_real": 0.6878,
"actg175|xgb|smote|auroc_new": 0.9933,
"actg175|xgb|smote_round|auroc_pooled": 0.8395,
"actg175|xgb|smote_round|auroc_real": 0.7244,
"actg175|xgb|smote_round|auroc_new": 0.8941,
"cdc_dhi|lr|none|auroc_pooled": 0.8225,
"cdc_dhi|lr|none|auroc_real": 0.8225,
"cdc_dhi|lr|postsplit_both_separately|auroc_pooled": 0.8308,
"cdc_dhi|lr|postsplit_both_separately|auroc_real": 0.8204,
"cdc_dhi|lr|postsplit_both_separately|auroc_new": 0.8328,
"cdc_dhi|lr|ros|auroc_pooled": 0.8226,
"cdc_dhi|lr|ros|auroc_real": 0.8226,
"cdc_dhi|lr|ros|auroc_new": 0.8226,
"cdc_dhi|lr|smote|auroc_pooled": 0.83,
"cdc_dhi|lr|smote|auroc_real": 0.8204,
"cdc_dhi|lr|smote|auroc_new": 0.8318,
"cdc_dhi|lr|smote_round|auroc_pooled": 0.8276,
"cdc_dhi|lr|smote_round|auroc_real": 0.821,
"cdc_dhi|lr|smote_round|auroc_new": 0.8289,
"cdc_dhi|xgb|none|auroc_pooled": 0.8307,
"cdc_dhi|xgb|none|auroc_real": 0.8307,
"cdc_dhi|xgb|postsplit_both_separately|auroc_pooled": 0.9707,
"cdc_dhi|xgb|postsplit_both_separately|auroc_real": 0.8232,
"cdc_dhi|xgb|postsplit_both_separately|auroc_new": 0.9992,
"cdc_dhi|xgb|ros|auroc_pooled": 0.8341,
"cdc_dhi|xgb|ros|auroc_real": 0.8338,
"cdc_dhi|xgb|ros|auroc_new": 0.8341,
"cdc_dhi|xgb|smote|auroc_pooled": 0.9692,
"cdc_dhi|xgb|smote|auroc_real": 0.8232,
"cdc_dhi|xgb|smote|auroc_new": 0.9974,
"cdc_dhi|xgb|smote_round|auroc_pooled": 0.8401,
"cdc_dhi|xgb|smote_round|auroc_real": 0.8313,
"cdc_dhi|xgb|smote_round|auroc_new": 0.8418,
"ctg|lr|none|auroc_pooled": 0.9861,
"ctg|lr|none|auroc_real": 0.9861,
"ctg|lr|postsplit_both_separately|auroc_pooled": 0.9903,
"ctg|lr|postsplit_both_separately|auroc_real": 0.9861,
"ctg|lr|postsplit_both_separately|auroc_new": 0.9907,
"ctg|lr|ros|auroc_pooled": 0.9898,
"ctg|lr|ros|auroc_real": 0.9897,
"ctg|lr|ros|auroc_new": 0.9898,
"ctg|lr|smote|auroc_pooled": 0.9919,
"ctg|lr|smote|auroc_real": 0.9893,
"ctg|lr|smote|auroc_new": 0.9922,
"ctg|lr|smote_round|auroc_pooled": 0.9919,
"ctg|lr|smote_round|auroc_real": 0.9893,
"ctg|lr|smote_round|auroc_new": 0.9921,
"ctg|xgb|none|auroc_pooled": 0.9966,
"ctg|xgb|none|auroc_real": 0.9966,
"ctg|xgb|postsplit_both_separately|auroc_pooled": 0.9984,
"ctg|xgb|postsplit_both_separately|auroc_real": 0.9965,
"ctg|xgb|postsplit_both_separately|auroc_new": 0.9986,
"ctg|xgb|ros|auroc_pooled": 0.9997,
"ctg|xgb|ros|auroc_real": 0.9997,
"ctg|xgb|ros|auroc_new": 0.9997,
"ctg|xgb|smote|auroc_pooled": 0.9996,
"ctg|xgb|smote|auroc_real": 0.9987,
"ctg|xgb|smote|auroc_new": 0.9997,
"ctg|xgb|smote_round|auroc_pooled": 0.9995,
"ctg|xgb|smote_round|auroc_real": 0.9988,
"ctg|xgb|smote_round|auroc_new": 0.9996,
"heart_failure|lr|none|auroc_pooled": 0.7586,
"heart_failure|lr|none|auroc_real": 0.7586,
"heart_failure|lr|postsplit_both_separately|auroc_pooled": 0.7732,
"heart_failure|lr|postsplit_both_separately|auroc_real": 0.7539,
"heart_failure|lr|postsplit_both_separately|auroc_new": 0.7905,
"heart_failure|lr|ros|auroc_pooled": 0.7823,
"heart_failure|lr|ros|auroc_real": 0.7788,
"heart_failure|lr|ros|auroc_new": 0.7854,
"heart_failure|lr|smote|auroc_pooled": 0.8018,
"heart_failure|lr|smote|auroc_real": 0.7811,
"heart_failure|lr|smote|auroc_new": 0.8203,
"heart_failure|lr|smote_round|auroc_pooled": 0.8009,
"heart_failure|lr|smote_round|auroc_real": 0.7813,
"heart_failure|lr|smote_round|auroc_new": 0.8185,
"heart_failure|xgb|none|auroc_pooled": 0.7417,
"heart_failure|xgb|none|auroc_real": 0.7417,
"heart_failure|xgb|postsplit_both_separately|auroc_pooled": 0.826,
"heart_failure|xgb|postsplit_both_separately|auroc_real": 0.7356,
"heart_failure|xgb|postsplit_both_separately|auroc_new": 0.9071,
"heart_failure|xgb|ros|auroc_pooled": 0.8631,
"heart_failure|xgb|ros|auroc_real": 0.8344,
"heart_failure|xgb|ros|auroc_new": 0.8887,
"heart_failure|xgb|smote|auroc_pooled": 0.8756,
"heart_failure|xgb|smote|auroc_real": 0.7725,
"heart_failure|xgb|smote|auroc_new": 0.9681,
"heart_failure|xgb|smote_round|auroc_pooled": 0.8519,
"heart_failure|xgb|smote_round|auroc_real": 0.7885,
"heart_failure|xgb|smote_round|auroc_new": 0.9089,
"mammography|lr|none|auroc_pooled": 0.9214,
"mammography|lr|none|auroc_real": 0.9214,
"mammography|lr|postsplit_both_separately|auroc_pooled": 0.9317,
"mammography|lr|postsplit_both_separately|auroc_real": 0.9222,
"mammography|lr|postsplit_both_separately|auroc_new": 0.9319,
"mammography|lr|ros|auroc_pooled": 0.9281,
"mammography|lr|ros|auroc_real": 0.9274,
"mammography|lr|ros|auroc_new": 0.9281,
"mammography|lr|smote|auroc_pooled": 0.9316,
"mammography|lr|smote|auroc_real": 0.9262,
"mammography|lr|smote|auroc_new": 0.9318,
"mammography|lr|smote_round|auroc_pooled": 0.9316,
"mammography|lr|smote_round|auroc_real": 0.9262,
"mammography|lr|smote_round|auroc_new": 0.9318,
"mammography|xgb|none|auroc_pooled": 0.9453,
"mammography|xgb|none|auroc_real": 0.9453,
"mammography|xgb|postsplit_both_separately|auroc_pooled": 0.9688,
"mammography|xgb|postsplit_both_separately|auroc_real": 0.9442,
"mammography|xgb|postsplit_both_separately|auroc_new": 0.9694,
"mammography|xgb|ros|auroc_pooled": 0.9905,
"mammography|xgb|ros|auroc_real": 0.9903,
"mammography|xgb|ros|auroc_new": 0.9905,
"mammography|xgb|smote|auroc_pooled": 0.99,
"mammography|xgb|smote|auroc_real": 0.9792,
"mammography|xgb|smote|auroc_new": 0.9903,
"mammography|xgb|smote_round|auroc_pooled": 0.99,
"mammography|xgb|smote_round|auroc_real": 0.9792,
"mammography|xgb|smote_round|auroc_new": 0.9903,
"thoracic|lr|none|auroc_pooled": 0.6431,
"thoracic|lr|none|auroc_real": 0.6431,
"thoracic|lr|postsplit_both_separately|auroc_pooled": 0.6174,
"thoracic|lr|postsplit_both_separately|auroc_real": 0.6184,
"thoracic|lr|postsplit_both_separately|auroc_new": 0.6172,
"thoracic|lr|ros|auroc_pooled": 0.7605,
"thoracic|lr|ros|auroc_real": 0.7479,
"thoracic|lr|ros|auroc_new": 0.7631,
"thoracic|lr|smote|auroc_pooled": 0.7588,
"thoracic|lr|smote|auroc_real": 0.7325,
"thoracic|lr|smote|auroc_new": 0.7644,
"thoracic|lr|smote_round|auroc_pooled": 0.7393,
"thoracic|lr|smote_round|auroc_real": 0.7303,
"thoracic|lr|smote_round|auroc_new": 0.7412,
"thoracic|xgb|none|auroc_pooled": 0.6016,
"thoracic|xgb|none|auroc_real": 0.6016,
"thoracic|xgb|postsplit_both_separately|auroc_pooled": 0.9082,
"thoracic|xgb|postsplit_both_separately|auroc_real": 0.6174,
"thoracic|xgb|postsplit_both_separately|auroc_new": 0.9699,
"thoracic|xgb|ros|auroc_pooled": 0.9287,
"thoracic|xgb|ros|auroc_real": 0.9062,
"thoracic|xgb|ros|auroc_new": 0.9334,
"thoracic|xgb|smote|auroc_pooled": 0.9455,
"thoracic|xgb|smote|auroc_real": 0.7267,
"thoracic|xgb|smote|auroc_new": 0.9919,
"thoracic|xgb|smote_round|auroc_pooled": 0.8695,
"thoracic|xgb|smote_round|auroc_real": 0.8205,
"thoracic|xgb|smote_round|auroc_new": 0.8799,
"simulation_binary_n1000|lr|none|auroc_pooled": 0.6505,
"simulation_binary_n1000|lr|none|auroc_real": 0.6505,
"simulation_binary_n1000|lr|postsplit_both_separately|auroc_pooled": 0.6654,
"simulation_binary_n1000|lr|postsplit_both_separately|auroc_real": 0.6471,
"simulation_binary_n1000|lr|postsplit_both_separately|auroc_new": 0.668,
"simulation_binary_n1000|lr|smote|auroc_pooled": 0.7418,
"simulation_binary_n1000|lr|smote|auroc_real": 0.7078,
"simulation_binary_n1000|lr|smote|auroc_new": 0.7466,
"simulation_binary_n1000|xgb|none|auroc_pooled": 0.5862,
"simulation_binary_n1000|xgb|none|auroc_real": 0.5862,
"simulation_binary_n1000|xgb|postsplit_both_separately|auroc_pooled": 0.9482,
"simulation_binary_n1000|xgb|postsplit_both_separately|auroc_real": 0.5997,
"simulation_binary_n1000|xgb|postsplit_both_separately|auroc_new": 0.9983,
"simulation_binary_n1000|xgb|smote|auroc_pooled": 0.9642,
"simulation_binary_n1000|xgb|smote|auroc_real": 0.7179,
"simulation_binary_n1000|xgb|smote|auroc_new": 0.9997,
"simulation_continuous_n1000|lr|none|auroc_pooled": 0.6572,
"simulation_continuous_n1000|lr|none|auroc_real": 0.6572,
"simulation_continuous_n1000|lr|postsplit_both_separately|auroc_pooled": 0.6603,
"simulation_continuous_n1000|lr|postsplit_both_separately|auroc_real": 0.6505,
"simulation_continuous_n1000|lr|postsplit_both_separately|auroc_new": 0.6616,
"simulation_continuous_n1000|lr|smote|auroc_pooled": 0.7413,
"simulation_continuous_n1000|lr|smote|auroc_real": 0.7177,
"simulation_continuous_n1000|lr|smote|auroc_new": 0.7444,
"simulation_continuous_n1000|xgb|none|auroc_pooled": 0.6107,
"simulation_continuous_n1000|xgb|none|auroc_real": 0.6107,
"simulation_continuous_n1000|xgb|postsplit_both_separately|auroc_pooled": 0.7174,
"simulation_continuous_n1000|xgb|postsplit_both_separately|auroc_real": 0.6006,
"simulation_continuous_n1000|xgb|postsplit_both_separately|auroc_new": 0.7327,
"simulation_continuous_n1000|xgb|smote|auroc_pooled": 0.9635,
"simulation_continuous_n1000|xgb|smote|auroc_real": 0.9103,
"simulation_continuous_n1000|xgb|smote|auroc_new": 0.9705,
"simulation_binary_n10000|lr|none|auroc_pooled": 0.6962,
"simulation_binary_n10000|lr|none|auroc_real": 0.6962,
"simulation_binary_n10000|lr|postsplit_both_separately|auroc_pooled": 0.722,
"simulation_binary_n10000|lr|postsplit_both_separately|auroc_real": 0.6958,
"simulation_binary_n10000|lr|postsplit_both_separately|auroc_new": 0.7256,
"simulation_binary_n10000|lr|smote|auroc_pooled": 0.7284,
"simulation_binary_n10000|lr|smote|auroc_real": 0.7025,
"simulation_binary_n10000|lr|smote|auroc_new": 0.7319,
"simulation_binary_n10000|xgb|none|auroc_pooled": 0.6716,
"simulation_binary_n10000|xgb|none|auroc_real": 0.6716,
"simulation_binary_n10000|xgb|postsplit_both_separately|auroc_pooled": 0.962,
"simulation_binary_n10000|xgb|postsplit_both_separately|auroc_real": 0.6805,
"simulation_binary_n10000|xgb|postsplit_both_separately|auroc_new": 1.0,
"simulation_binary_n10000|xgb|smote|auroc_pooled": 0.9655,
"simulation_binary_n10000|xgb|smote|auroc_real": 0.7096,
"simulation_binary_n10000|xgb|smote|auroc_new": 1.0,
"simulation_continuous_n10000|lr|none|auroc_pooled": 0.6917,
"simulation_continuous_n10000|lr|none|auroc_real": 0.6917,
"simulation_continuous_n10000|lr|postsplit_both_separately|auroc_pooled": 0.7019,
"simulation_continuous_n10000|lr|postsplit_both_separately|auroc_real": 0.6909,
"simulation_continuous_n10000|lr|postsplit_both_separately|auroc_new": 0.7033,
"simulation_continuous_n10000|lr|smote|auroc_pooled": 0.7117,
"simulation_continuous_n10000|lr|smote|auroc_real": 0.6983,
"simulation_continuous_n10000|lr|smote|auroc_new": 0.7135,
"simulation_continuous_n10000|xgb|none|auroc_pooled": 0.6561,
"simulation_continuous_n10000|xgb|none|auroc_real": 0.6561,
"simulation_continuous_n10000|xgb|postsplit_both_separately|auroc_pooled": 0.7671,
"simulation_continuous_n10000|xgb|postsplit_both_separately|auroc_real": 0.6311,
"simulation_continuous_n10000|xgb|postsplit_both_separately|auroc_new": 0.7852,
"simulation_continuous_n10000|xgb|smote|auroc_pooled": 0.8645,
"simulation_continuous_n10000|xgb|smote|auroc_real": 0.7577,
"simulation_continuous_n10000|xgb|smote|auroc_new": 0.8788,
"diabetes130|offsupport|n_new": 75948,
"diabetes130|offsupport|share_rows": 1.0,
"diabetes130|offsupport|median_features": 15.0
}
import json, numpy as np, pandas as pd
R = os.path.join(WORK, 'results')
cols = ['auroc_pooled', 'auroc_real', 'auroc_new']
got = {}
def add(df, keys, tag):
    for k, g in df.groupby(keys):
        k = k if isinstance(k, tuple) else (k,)
        for c in cols:
            v = g[c].mean()
            if pd.notna(v): got['|'.join([tag] + [str(x) for x in k] + [c])] = float(v)
for f in ('mechanism_primary', 'mechanism_postsplit', 'mechanism_round_binary'):
    add(pd.read_csv(f'{R}/{f}.csv'), ['learner', 'method'], 'diabetes130')
rep = pd.read_csv(f'{R}/replication_datasets.csv')
for ds, g in rep.groupby('dataset'): add(g, ['learner', 'method'], ds)
sim = pd.read_csv(f'{R}/simulation.csv')
for (n, kind), g in sim.groupby(['n', 'kind']): add(g, ['learner', 'method'], f'simulation_{kind}_n{n}')
off = json.load(open(f'{R}/off_support_primary.json'))
got['diabetes130|offsupport|n_new'] = off['n_new']
got['diabetes130|offsupport|share_rows'] = off['share_rows_off_support']
got['diabetes130|offsupport|median_features'] = off['median_off_support_features']

rows = []
for k, e in EXPECTED.items():
    g = got.get(k, np.nan); d = g - e if pd.notna(g) else np.nan
    tol = 0 if 'offsupport' in k else 1
    if pd.isna(d): v = 'MISSING'
    elif abs(d) <= (0 if tol == 0 else 0.005): v = 'same'
    elif abs(d) <= 0.02: v = 'small difference'
    else: v = 'DIFFERENT'
    rows.append(dict(item=k, manuscript=e, yours=None if pd.isna(g) else round(g, 4), difference=None if pd.isna(d) else round(d, 4), verdict=v))
cmp = pd.DataFrame(rows)
cmp.to_csv(os.path.join(WORK, 'verify_comparison.csv'), index=False)
pd.set_option('display.max_rows', 500, 'display.width', 200)
print(cmp.verdict.value_counts().to_string(), '\n')
bad = cmp[cmp.verdict != 'same']
if len(bad) == 0:
    print('VERDICT: every number matches the manuscript to within 0.005.')
else:
    print('Items that are not the same:'); print(bad.to_string(index=False))
    print('\nVERDICT: send verify_comparison.csv to Claude before submitting.')
print('\nSaved:', os.path.join(WORK, 'verify_comparison.csv'))